# COMPARACIÓN DE MODELOS SOBRE TEST SET — CNN vs Vision Transformer

Evaluación y **comparación** de dos cascadas clínicas sobre **validación y test** (contraste homogéneo, mismo pipeline calibrado):

| | Modelo **B34** (CNN) | Modelo **DINO** (Transformer) |
|---|---|---|
| Triaje binario | EfficientNet-B3 (300px) | DINOv2 ViT-L (294px) |
| Especialista | EfficientNet-B4 (380px) | DINOv2 ViT-L (378px) |

Produce, sobre val y test: tablas comparativas + **DeLong** (¿diferencia de AUC significativa?) + **McNemar** (¿uno detecta más melanomas?) + **equidad por subgrupo** (sexo / edad / sitio) + matrices de confusión + JSONs.

In [1]:
# ============================================================
# Preparación del entorno en Google Colab
# ============================================================
# 1) Clona el código del proyecto (src/, scripts/, requirements.txt).
# 2) Instala las dependencias.
# 3) Monta Google Drive (datos y checkpoints).
# 4) Descomprime el dataset (imágenes + CSVs) al disco local de Colab.
# 5) Enlaza data/ (lectura) y models/ (escritura en Drive).
#
# Requisitos previos en tu Google Drive (MyDrive/SKIN-PROYECT/):
#   - skin_data_cloud.zip   (dataset: contiene data/raw y data/processed)
#   - models/               (checkpoints; se crea solo si no existe)

import os
import sys
import glob
import subprocess

REPO_URL = "https://github.com/Super1232/SKIN-MOTOR-URB.git"  # URL de tu repositorio
PROJECT_ROOT = "/content/SKIN-PROYECT"
DRIVE_ROOT = "/content/drive/MyDrive/SKIN-PROYECT"

# 1. Código del proyecto
if not os.path.exists(os.path.join(PROJECT_ROOT, "src")):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, PROJECT_ROOT])
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
print(f"Código del proyecto en: {PROJECT_ROOT}")

# 2. Dependencias
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# 3. Google Drive
from google.colab import drive
try:
    drive.mount("/content/drive", force_remount=False)
    print("Google Drive montado en /content/drive")
except Exception as e:
    raise RuntimeError(
        f"No se pudo montar Google Drive: {e}\n"
        "Verifica: 1) aceptar el prompt de autorización, 2) conexión de red, "
        "3) si el runtime se reinició, reinicia el kernel y ejecuta de nuevo."
    ) from e

# 4. Dataset: descomprimir el zip al disco local de Colab (imágenes + CSVs)
zip_candidates = [
    f"{DRIVE_ROOT}/skin_data_cloud.zip",
    "/content/drive/MyDrive/skin_data_cloud.zip",
]
ZIP_PATH = next((p for p in zip_candidates if os.path.exists(p)), None)
if ZIP_PATH is None:
    raise FileNotFoundError(
        "No se encuentra 'skin_data_cloud.zip' en tu Drive "
        f"(buscado en: {zip_candidates})."
    )
if not os.path.exists("/content/data/raw"):
    print("Descomprimiendo el dataset al disco local (puede tardar unos minutos)...")
    subprocess.check_call(["unzip", "-q", "-n", ZIP_PATH, "-d", "/content"])

# 5. Enlaces: data/ (raw + processed) y models/ (en Drive)
os.makedirs("data", exist_ok=True)
for sub in ("raw", "processed"):
    origen = f"/content/data/{sub}"
    destino = f"data/{sub}"
    if os.path.exists(origen) and not os.path.exists(destino):
        os.symlink(origen, destino)
if not glob.glob("data/raw/*"):
    raise RuntimeError(
        "No se encontraron datasets en data/raw tras descomprimir. "
        "Revisa que skin_data_cloud.zip contenga data/raw/."
    )

os.makedirs(f"{DRIVE_ROOT}/models/checkpoints", exist_ok=True)
if not os.path.exists("models"):
    os.symlink(f"{DRIVE_ROOT}/models", "models")

print("Entorno listo.")

Mounted at /content/drive
Google Drive montado en /content/drive
Descomprimiendo el dataset al disco local (puede tardar unos minutos)...
Entorno listo.


## Configuración y utilidades

In [2]:
# === SETUP: imports, configuración, helpers y tests estadísticos ===
import os, json, math, warnings
import numpy as np
import pandas as pd
import torch
from scipy import stats
import matplotlib


matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

from scripts.calibrate_temperature import _build_and_load
from src.data.dataset import get_dataloaders
from src.training.train import _tta_forward_views
from src.inference.calibration import TemperatureScaler
from src.inference.predict import _detect_backbone_config, ClinicalInferenceEngine
from src.data.constants import (
    CLINICAL_THRESHOLD_BINARY, MULTICLASS_INDEX_MAP, MULTICLASS_LABEL_MAP,
)
from src.utils.evaluation import (
    expected_calibration_error, partial_auc, partial_auc_above_sensitivity,
    sensitivity_at_specificity, brier_score, fairness_report,
    plot_confusion_matrix, plot_per_class_pr_curves, plot_reliability_diagram,
)
from sklearn.metrics import roc_auc_score, recall_score

# ----------------------------------------------------------------------
# CONFIGURACIÓN
# ----------------------------------------------------------------------
MODELS = {
    "B34 (CNN)":    "models/B34",
    "DINO (ViT-L)": "models/DINO-V2",
}
CKPT_TAG         = "SOUP"   # SOUP (recomendado) | SWA | EMA | "" (best puro)
TTA_VIEWS        = 4        # vistas TTA para las metricas (1 / 4 / 6). 4 = como en training
BATCH_SIZE       = 32       # A100 40GB aguanta mas; conservador para ViT-L 378px
N_CASCADE_SAMPLE = 400      # imagenes para la Parte 3 (cascada per-image, mas lenta)
SEED             = 42
OUT_DIR          = "/content/drive/MyDrive/SKIN-PROYECT/evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = (torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported())
             else torch.float16)
print(f"Device: {device} | AMP: {amp_dtype} | TTA: {TTA_VIEWS} vistas | "
      f"Checkpoint: {CKPT_TAG or 'best'}")

# ----------------------------------------------------------------------
# Helpers de rutas / checkpoints
# ----------------------------------------------------------------------
def ckpt_name(mode, tag):
    base = f"best_{mode}_model"
    return f"{base}_{tag}.pth" if tag else f"{base}.pth"

def find_ckpt(folder, mode):
    """Ruta del checkpoint preferido (CKPT_TAG) con fallback a best puro."""
    p = os.path.join(folder, ckpt_name(mode, CKPT_TAG))
    if os.path.exists(p):
        return p
    p2 = os.path.join(folder, ckpt_name(mode, ""))
    if os.path.exists(p2):
        print(f"  AVISO: {p} no existe; uso {p2}")
        return p2
    return None

def opt(path):
    """Devuelve path si existe, None si no (para calibradores opcionales del engine)."""
    return path if (path and os.path.exists(path)) else None

def detect_kind(ckpt_path):
    """Devuelve (kind, backbone) leyendo el state_dict. kind in {cnn, vit}."""
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    if isinstance(sd, dict) and "model_state_dict" in sd:
        sd = sd["model_state_dict"]
    bb, _ = _detect_backbone_config(sd)
    kind = "vit" if (bb and ("vit" in bb or "dinov2" in bb)) else "cnn"
    return kind, bb

print("\n" + "=" * 64)
MODEL_META = {}
for name, folder in MODELS.items():
    binp = find_ckpt(folder, "binary")
    mulp = find_ckpt(folder, "multiclass")
    if binp is None:
        raise FileNotFoundError(
            f"ERROR: {name}: no encuentro el binario en '{folder}'. "
            f"Sube la carpeta a tu Drive en SKIN-PROYECT/{folder} "
            f"(o ajusta MODELS arriba).")
    kind, bb_bin = detect_kind(binp)
    bb_mul = detect_kind(mulp)[1] if mulp else None
    MODEL_META[name] = dict(folder=folder, binary=binp, multiclass=mulp,
                            kind=kind, backbone_bin=bb_bin, backbone_mul=bb_mul)
    print(f"{name:14s} | kind={kind} | bin={bb_bin} | mul={bb_mul} | "
          f"multiclase={'si' if mulp else 'NO'}")
print("=" * 64)

# ----------------------------------------------------------------------
# Inferencia batched con TTA (logits crudos + labels)
# ----------------------------------------------------------------------
def collect_test_logits(model, loader, num_views=TTA_VIEWS, desc="eval"):
    all_logits, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels, meta in tqdm(loader, desc=desc, leave=False):
            images = images.to(device, non_blocking=True)
            meta = meta.to(device, non_blocking=True)
            logits = _tta_forward_views(model, images, meta, num_views, amp_dtype)
            all_logits.append(logits.float().cpu())
            all_labels.append(labels.cpu())
    return torch.cat(all_logits, 0), torch.cat(all_labels, 0)

def load_temperature(folder, mode, num_classes):
    p = os.path.join(folder, f"temperature_{mode}.pt")
    if not os.path.exists(p):
        return None
    try:
        ts = TemperatureScaler(num_classes=num_classes)
        ts.load(p)
        return ts
    except Exception as e:
        print(f"  AVISO: no pude cargar {p}: {e}")
        return None

# ----------------------------------------------------------------------
# Tests estadísticos: DeLong (compara dos AUC) y McNemar
# ----------------------------------------------------------------------
def _midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=np.float64)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1.0
        i = j
    T2 = np.empty(N, dtype=np.float64)
    T2[J] = T
    return T2

def delong_test(y_true, score1, score2):
    """DeLong (Sun & Xu 2014): compara dos AUC sobre las MISMAS muestras.
    H0: AUC1 == AUC2. Devuelve auc1, auc2, z, p_value."""
    y_true = np.asarray(y_true).astype(int)
    s = np.vstack((np.asarray(score1, dtype=np.float64),
                   np.asarray(score2, dtype=np.float64)))
    order = (-y_true).argsort(kind="mergesort")   # positivos primero
    s = s[:, order]
    m = int(y_true.sum())
    n = s.shape[1] - m
    if m == 0 or n == 0:
        return dict(auc1=float("nan"), auc2=float("nan"),
                    z=float("nan"), p_value=float("nan"), note="falta una clase")
    pos, neg = s[:, :m], s[:, m:]
    k = s.shape[0]
    tx = np.vstack([_midrank(pos[r]) for r in range(k)])
    ty = np.vstack([_midrank(neg[r]) for r in range(k)])
    tz = np.vstack([_midrank(s[r]) for r in range(k)])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    cov = sx / m + sy / n
    L = np.array([[1.0, -1.0]])
    var = float(L @ cov @ L.T)
    if var <= 0:
        z, p = 0.0, 1.0
    else:
        z = float(np.abs(aucs[0] - aucs[1]) / np.sqrt(var))
        p = float(2.0 * (1.0 - stats.norm.cdf(z)))
    return dict(auc1=float(aucs[0]), auc2=float(aucs[1]), z=z, p_value=p)

def mcnemar_test(y_true, pred1, pred2):
    """McNemar exacto (binomial two-sided) sobre decisiones binarias 0/1.
    b = m1 acierta & m2 falla; c = m1 falla & m2 acierta."""
    y_true = np.asarray(y_true).astype(int)
    c1 = (np.asarray(pred1).astype(int) == y_true)
    c2 = (np.asarray(pred2).astype(int) == y_true)
    b = int(np.sum(c1 & ~c2))
    c = int(np.sum(~c1 & c2))
    nd = b + c
    if nd == 0:
        return dict(b=b, c=c, n_discordant=0, p_value=1.0)
    p = float(min(1.0, 2.0 * stats.binom.cdf(min(b, c), nd, 0.5)))
    return dict(b=b, c=c, n_discordant=nd, p_value=p)

def fmt_p(p):
    if p is None or (isinstance(p, float) and p != p):
        return "n/a"
    return "<0.001" if p < 0.001 else f"{p:.3f}"

print("\nSetup completo. Ejecuta la Parte 1 (binario).")

Device: cuda | AMP: torch.bfloat16 | TTA: 4 vistas | Checkpoint: SOUP

B34 (CNN)      | kind=cnn | bin=effnet_b3 | mul=effnet_b4 | multiclase=si
DINO (ViT-L)   | kind=vit | bin=dinov2_vitl14 | mul=dinov2_vitl14 | multiclase=si

Setup completo. Ejecuta la Parte 1 (binario).


## Parte 1 — Triaje binario: B3 VS ViT-L (val + test)

Evalúa el triaje binario de cada modelo sobre **validación y test** con el mismo pipeline calibrado (contraste homogéneo) y los enfrenta con **DeLong** (AUC) y **McNemar** (decisiones al umbral clínico 0.20). La métrica clave es el **Recall@0.20** (sensibilidad: melanomas no perdidos). Incluye **equidad por sexo / edad / sitio**.

In [3]:
# === PARTE 1: COMPARACIÓN DEL TRIAJE BINARIO sobre val Y test ===
# Cada modelo se evalúa sobre 04_{split}.csv (calibrado), en val y test, para un
# contraste homogéneo (mismo pipeline). Comparación DeLong (AUC) + McNemar (@0.20)
# por split + equidad (fairness_report) por sexo/edad/sitio.

SPLITS = ["val", "test"]

def get_split_loader(mode, kind, split):
    """Loader de val o test (sin augmentation, shuffle=False)."""
    _tr, val_l, test_l = get_dataloaders(
        mode=mode, batch_size=BATCH_SIZE, backbone_kind=kind, seed=SEED)
    return val_l if split == "val" else test_l

bin_results, bin_probs, bin_labels = {}, {}, {}

for split in SPLITS:
    for name, meta in MODEL_META.items():
        key = f"{name} [{split}]"
        print(f"\n> Binario: {key}  (kind={meta['kind']}, backbone={meta['backbone_bin']})")
        model = _build_and_load("binary", meta["binary"], device)
        loader = get_split_loader("binary", meta["kind"], split)
        logits, labels = collect_test_logits(model, loader, TTA_VIEWS, desc=f"bin:{key}")
        logits = logits.view(len(labels), -1)
        y = labels.numpy().astype(int)

        prob_raw = torch.sigmoid(logits.squeeze(-1)).detach().numpy()
        ts = load_temperature(meta["folder"], "binary", num_classes=1)
        prob_cal = (torch.sigmoid(ts.calibrate(logits).squeeze(-1)).detach().numpy()
                    if ts is not None else prob_raw)

        thr = CLINICAL_THRESHOLD_BINARY
        pred = (prob_cal >= thr).astype(int)
        tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
        fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
        recall = tp / (tp + fn) if (tp + fn) else float("nan")
        spec = tn / (tn + fp) if (tn + fp) else float("nan")
        sens95 = sensitivity_at_specificity(y, prob_cal, 0.95).get("sensitivity")

        meta_df = pd.read_csv(f"data/processed/04_{split}.csv")[
            ["age", "sex", "anatom_site"]].rename(columns={"age": "age_approx"})
        fair = fairness_report(y, prob_cal, meta_df, mode="binary", threshold=thr)

        res = dict(
            model=name, split=split, n=int(len(y)), n_malignant=int(y.sum()),
            auc=float(roc_auc_score(y, prob_cal)),
            pauc_sens80=float(partial_auc_above_sensitivity(y, prob_cal, 0.80)),
            pauc_spec95=float(partial_auc(y, prob_cal, 0.05)),
            recall_at_020=float(recall), specificity_at_020=float(spec),
            sens_at_spec95=(float(sens95) if sens95 is not None else None),
            ece_raw=float(expected_calibration_error(prob_raw, y)[0]),
            ece_cal=float(expected_calibration_error(prob_cal, y)[0]),
            brier=float(brier_score(y, prob_cal)),
            tp=tp, tn=tn, fp=fp, fn=fn, threshold=float(thr),
            backbone=meta["backbone_bin"], kind=meta["kind"],
            fairness=fair,
        )
        bin_results[key] = res
        bin_probs[(split, name)] = prob_cal
        bin_labels[(split, name)] = y
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
        print(f"  AUC={res['auc']:.4f} | Recall@0.20={recall:.4f} | Spec@0.20={spec:.4f} "
              f"| melanomas perdidos (FN)={fn}/{int(y.sum())}")
        for fl in fair["bias_flags"]:
            print("    sesgo:", fl)

# --- Tabla comparativa (val + test) ---
_cols = ["model", "split", "backbone", "auc", "pauc_sens80", "recall_at_020",
         "specificity_at_020", "sens_at_spec95", "ece_cal", "brier", "fn", "n_malignant"]
df_bin = pd.DataFrame({k: {c: v[c] for c in _cols} for k, v in bin_results.items()}).T
print("\n" + "=" * 78 + "\nCOMPARATIVA BINARIA (val + test)\n" + "=" * 78)
print(df_bin.to_string())

# --- DeLong + McNemar por split (entre los 2 modelos, mismas imágenes) ---
names = list(MODEL_META.keys())
stat = {}
if len(names) == 2:
    a, b = names
    for split in SPLITS:
        ya, yb = bin_labels[(split, a)], bin_labels[(split, b)]
        if not np.array_equal(ya, yb):
            print(f"\n  AVISO [{split}]: etiquetas no alineadas indice a indice; tests omitidos.")
            continue
        d = delong_test(ya, bin_probs[(split, a)], bin_probs[(split, b)])
        pa = (bin_probs[(split, a)] >= CLINICAL_THRESHOLD_BINARY).astype(int)
        pb = (bin_probs[(split, b)] >= CLINICAL_THRESHOLD_BINARY).astype(int)
        mc = mcnemar_test(ya, pa, pb)
        stat[split] = dict(modelo_1=a, modelo_2=b, delong=d, mcnemar=mc)
        sig = "DIFERENCIA SIGNIFICATIVA" if d["p_value"] < 0.05 else "sin diferencia significativa"
        print(f"\n  [{split}] DeLong (AUC): {a}={d['auc1']:.4f} vs {b}={d['auc2']:.4f} "
              f"| p={fmt_p(d['p_value'])} -> {sig}")
        print(f"  [{split}] McNemar @0.20: aciertos exclusivos {a}={mc['b']} | {b}={mc['c']} "
              f"| discordantes={mc['n_discordant']} | p={fmt_p(mc['p_value'])}")

# --- Reliability diagrams por modelo y split ---
for split in SPLITS:
    for name in names:
        short = name.split()[0]
        plot_reliability_diagram(
            bin_probs[(split, name)], bin_labels[(split, name)],
            title=f"Reliability - {name} [{split}] (binario)",
            save_path=os.path.join(OUT_DIR, f"reliability_binary_{short}_{split}.png"))

# --- Guardar JSON ---
out = dict(
    config=dict(tta=TTA_VIEWS, ckpt=CKPT_TAG, threshold=float(CLINICAL_THRESHOLD_BINARY), splits=SPLITS),
    per_model=bin_results, statistical_tests=stat,
)
with open(os.path.join(OUT_DIR, "comparison_binary.json"), "w") as f:
    json.dump(out, f, indent=2, default=str)
print(f"\nGuardado: {OUT_DIR}/comparison_binary.json  (+ reliability PNGs val/test)")


> Binario: B34 (CNN) [val]  (kind=cnn, backbone=effnet_b3)
[INFO] Usando 8 procesos de carga de datos.


bin:B34 (CNN) [val]:   0%|          | 0/1344 [00:00<?, ?it/s]

  AUC=0.9852 | Recall@0.20=0.9628 | Spec@0.20=0.9583 | melanomas perdidos (FN)=37/995
    sesgo: SESGO POTENCIAL por edad: gap AUC = 0.100 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap specificity = 0.101 (> 0.05, threshold clínico=0.2)
    sesgo: SESGO POTENCIAL por edad: gap recall = 0.237 (> 0.05, threshold clínico=0.2)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap AUC = 0.096 (> 0.05)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap specificity = 0.647 (> 0.05, threshold clínico=0.2)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap recall = 0.120 (> 0.05, threshold clínico=0.2)

> Binario: DINO (ViT-L) [val]  (kind=vit, backbone=dinov2_vitl14)
[INFO] Usando 8 procesos de carga de datos.


bin:DINO (ViT-L) [val]:   0%|          | 0/1344 [00:00<?, ?it/s]

  AUC=0.9879 | Recall@0.20=0.9678 | Spec@0.20=0.9479 | melanomas perdidos (FN)=32/995
    sesgo: SESGO POTENCIAL por edad: gap AUC = 0.120 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap specificity = 0.116 (> 0.05, threshold clínico=0.2)
    sesgo: SESGO POTENCIAL por edad: gap recall = 0.246 (> 0.05, threshold clínico=0.2)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap AUC = 0.156 (> 0.05)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap specificity = 0.966 (> 0.05, threshold clínico=0.2)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap recall = 0.072 (> 0.05, threshold clínico=0.2)

> Binario: B34 (CNN) [test]  (kind=cnn, backbone=effnet_b3)
[INFO] Usando 8 procesos de carga de datos.


bin:B34 (CNN) [test]:   0%|          | 0/1344 [00:00<?, ?it/s]

  AUC=0.9867 | Recall@0.20=0.9708 | Spec@0.20=0.9575 | melanomas perdidos (FN)=29/993
    sesgo: SESGO POTENCIAL por sitio anatómico: gap AUC = 0.169 (> 0.05)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap specificity = 0.937 (> 0.05, threshold clínico=0.2)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap recall = 0.181 (> 0.05, threshold clínico=0.2)

> Binario: DINO (ViT-L) [test]  (kind=vit, backbone=dinov2_vitl14)
[INFO] Usando 8 procesos de carga de datos.


bin:DINO (ViT-L) [test]:   0%|          | 0/1344 [00:00<?, ?it/s]

  AUC=0.9892 | Recall@0.20=0.9809 | Spec@0.20=0.9461 | melanomas perdidos (FN)=19/993
    sesgo: SESGO POTENCIAL por sitio anatómico: gap AUC = 0.448 (> 0.05)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap specificity = 0.963 (> 0.05, threshold clínico=0.2)
    sesgo: SESGO POTENCIAL por sitio anatómico: gap recall = 0.069 (> 0.05, threshold clínico=0.2)

COMPARATIVA BINARIA (val + test)
                            model split       backbone       auc pauc_sens80 recall_at_020 specificity_at_020 sens_at_spec95   ece_cal     brier  fn n_malignant
B34 (CNN) [val]         B34 (CNN)   val      effnet_b3  0.985156    0.967375      0.962814           0.958312       0.968844  0.059836  0.023777  37         995
DINO (ViT-L) [val]   DINO (ViT-L)   val  dinov2_vitl14  0.987899    0.973677      0.967839           0.947883       0.966834  0.060417  0.028732  32         995
B34 (CNN) [test]        B34 (CNN)  test      effnet_b3  0.986683    0.969157      0.970796           0.957499       0.974

## Parte 2 — Especialista multiclase: B4 VS ViT-L (val + test)

AUC macro y **por clase**, recall por clase, matrices de confusión enfrentadas, DeLong one-vs-rest por clase y **equidad por subgrupo**, sobre **validación y test**.

In [4]:
# === PARTE 2: COMPARACIÓN DEL ESPECIALISTA MULTICLASE sobre val Y test ===

SPLITS = ["val", "test"]

def get_split_loader(mode, kind, split):
    """Loader de val o test (sin augmentation, shuffle=False)."""
    _tr, val_l, test_l = get_dataloaders(
        mode=mode, batch_size=BATCH_SIZE, backbone_kind=kind, seed=SEED)
    return val_l if split == "val" else test_l

class_names = [MULTICLASS_INDEX_MAP[i] for i in range(len(MULTICLASS_INDEX_MAP))]
mul_results, mul_probs, mul_labels = {}, {}, {}

for split in SPLITS:
    for name, meta in MODEL_META.items():
        if meta["multiclass"] is None:
            print(f"\n> Multiclase: {name} [{split}] -- NO disponible, se omite.")
            continue
        key = f"{name} [{split}]"
        print(f"\n> Multiclase: {key}  (kind={meta['kind']}, backbone={meta['backbone_mul']})")
        model = _build_and_load("multiclass", meta["multiclass"], device)
        loader = get_split_loader("multiclass", meta["kind"], split)
        logits, labels = collect_test_logits(model, loader, TTA_VIEWS, desc=f"mul:{key}")
        y = labels.numpy().astype(int)
        ncls = len(class_names)

        ts = load_temperature(meta["folder"], "multiclass", num_classes=ncls)
        cal_logits = ts.calibrate(logits) if ts is not None else logits
        probs = torch.softmax(cal_logits, dim=1).detach().numpy()
        pred = probs.argmax(axis=1)

        auc_macro = float(roc_auc_score(y, probs, multi_class="ovr", average="macro"))
        auc_pc = roc_auc_score(y, probs, multi_class="ovr", average=None)
        rec_pc = recall_score(y, pred, average=None, labels=list(range(ncls)), zero_division=0)
        conf_top1 = probs.max(axis=1)

        meta_df = pd.read_csv(f"data/processed/07_{split}_multiclass.csv")[
            ["age", "sex", "anatom_site"]].rename(columns={"age": "age_approx"})
        fair = fairness_report(y, probs, meta_df, mode="multiclass", class_names=class_names)

        res = dict(
            model=name, split=split, n=int(len(y)), auc_macro=auc_macro,
            accuracy=float((pred == y).mean()),
            ece_top1=float(expected_calibration_error(conf_top1, (pred == y).astype(int))[0]),
            auc_per_class={class_names[i]: float(auc_pc[i]) for i in range(ncls)},
            recall_per_class={class_names[i]: float(rec_pc[i]) for i in range(ncls)},
            backbone=meta["backbone_mul"], kind=meta["kind"], fairness=fair,
        )
        mul_results[key] = res
        mul_probs[(split, name)] = probs
        mul_labels[(split, name)] = y

        short = name.split()[0]
        plot_confusion_matrix(
            y, pred, class_names, normalize=True,
            title=f"Matriz de confusion - {key}",
            save_path=os.path.join(OUT_DIR, f"confusion_{short}_{split}.png"))
        plot_per_class_pr_curves(
            y, probs, class_names,
            save_path=os.path.join(OUT_DIR, f"pr_curves_{short}_{split}.png"))
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
        print(f"  AUC macro={auc_macro:.4f} | Accuracy={res['accuracy']:.4f} "
              f"| ECE top-1={res['ece_top1']:.4f}")
        for fl in fair["bias_flags"][:6]:
            print("    sesgo:", fl)

# --- Tablas comparativas (por split) + DeLong one-vs-rest por clase ---
delong_pc = {}
if mul_results:
    names_m = list(MODEL_META.keys())
    for split in SPLITS:
        keys_split = [f"{n} [{split}]" for n in names_m if f"{n} [{split}]" in mul_results]
        if not keys_split:
            continue
        print("\n" + "=" * 78 + f"\nMULTICLASE [{split}] -- AUC (macro + por clase)\n" + "=" * 78)
        rows_auc = {k: {"AUC_macro": mul_results[k]["auc_macro"], **mul_results[k]["auc_per_class"]}
                    for k in keys_split}
        print(pd.DataFrame(rows_auc).T.to_string())
        print(f"\n--- Recall por clase [{split}] ---")
        rows_rec = {k: mul_results[k]["recall_per_class"] for k in keys_split}
        print(pd.DataFrame(rows_rec).T.to_string())

    if len(names_m) == 2:
        a, b = names_m
        for split in SPLITS:
            if (split, a) not in mul_probs or (split, b) not in mul_probs:
                continue
            if not np.array_equal(mul_labels[(split, a)], mul_labels[(split, b)]):
                print(f"\n  AVISO [{split}]: etiquetas multiclase no alineadas; DeLong omitido.")
                continue
            print(f"\n--- DeLong one-vs-rest por clase [{split}] ({a} vs {b}) ---")
            dd = {}
            for i, cn in enumerate(class_names):
                yb = (mul_labels[(split, a)] == i).astype(int)
                if yb.sum() == 0:
                    continue
                d = delong_test(yb, mul_probs[(split, a)][:, i], mul_probs[(split, b)][:, i])
                dd[cn] = d
                print(f"  {cn:20s}: {a}={d['auc1']:.3f} vs {b}={d['auc2']:.3f} | p={fmt_p(d['p_value'])}")
            delong_pc[split] = dd

    out = dict(
        config=dict(tta=TTA_VIEWS, ckpt=CKPT_TAG, classes=class_names, splits=SPLITS),
        per_model=mul_results, delong_per_class=delong_pc,
    )
    with open(os.path.join(OUT_DIR, "comparison_multiclass.json"), "w") as f:
        json.dump(out, f, indent=2, default=str)
    print(f"\nGuardado: {OUT_DIR}/comparison_multiclass.json  (+ confusion / PR PNGs val/test)")
else:
    print("\nAVISO: ningun modelo tiene multiclase; Parte 2 omitida.")


> Multiclase: B34 (CNN) [val]  (kind=cnn, backbone=effnet_b4)
[INFO] Usando 8 procesos de carga de datos.


mul:B34 (CNN) [val]:   0%|          | 0/94 [00:00<?, ?it/s]

  AUC macro=0.9424 | Accuracy=0.7684 | ECE top-1=0.0490
    sesgo: SESGO POTENCIAL por sexo: gap recall 'Basal Cell Carcinoma' = 0.087 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap macro AUC = 0.052 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Melanoma' = 0.440 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Nevus' = 0.064 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Basal Cell Carcinoma' = 0.072 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Benign Keratosis' = 0.318 (> 0.05)

> Multiclase: DINO (ViT-L) [val]  (kind=vit, backbone=dinov2_vitl14)
[INFO] Usando 8 procesos de carga de datos.


mul:DINO (ViT-L) [val]:   0%|          | 0/94 [00:00<?, ?it/s]

  AUC macro=0.9600 | Accuracy=0.8039 | ECE top-1=0.0510
    sesgo: SESGO POTENCIAL por sexo: gap recall 'Basal Cell Carcinoma' = 0.050 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Melanoma' = 0.478 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Nevus' = 0.077 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Basal Cell Carcinoma' = 0.105 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Benign Keratosis' = 0.203 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Squamous Cell Carcinoma' = 0.219 (> 0.05)

> Multiclase: B34 (CNN) [test]  (kind=cnn, backbone=effnet_b4)
[INFO] Usando 8 procesos de carga de datos.


mul:B34 (CNN) [test]:   0%|          | 0/94 [00:00<?, ?it/s]

  AUC macro=0.9482 | Accuracy=0.7592 | ECE top-1=0.0295
    sesgo: SESGO POTENCIAL por sexo: gap recall 'Melanoma' = 0.100 (> 0.05)
    sesgo: SESGO POTENCIAL por sexo: gap recall 'Squamous Cell Carcinoma' = 0.168 (> 0.05)
    sesgo: SESGO POTENCIAL por sexo: gap recall 'Actinic Keratosis' = 0.080 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Nevus' = 0.175 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Melanoma' = 0.200 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Basal Cell Carcinoma' = 0.093 (> 0.05)

> Multiclase: DINO (ViT-L) [test]  (kind=vit, backbone=dinov2_vitl14)
[INFO] Usando 8 procesos de carga de datos.


mul:DINO (ViT-L) [test]:   0%|          | 0/94 [00:00<?, ?it/s]

  AUC macro=0.9609 | Accuracy=0.8052 | ECE top-1=0.0509
    sesgo: SESGO POTENCIAL por sexo: gap recall 'Melanoma' = 0.117 (> 0.05)
    sesgo: SESGO POTENCIAL por sexo: gap recall 'Squamous Cell Carcinoma' = 0.212 (> 0.05)
    sesgo: SESGO POTENCIAL por sexo: gap recall 'Benign Keratosis' = 0.094 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Nevus' = 0.169 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Melanoma' = 0.225 (> 0.05)
    sesgo: SESGO POTENCIAL por edad: gap recall 'Benign Keratosis' = 0.160 (> 0.05)

MULTICLASE [val] -- AUC (macro + por clase)
                    AUC_macro  Melanoma     Nevus  Basal Cell Carcinoma  Squamous Cell Carcinoma  Benign Keratosis  Actinic Keratosis  Vascular Lesion  Dermatofibroma
B34 (CNN) [val]      0.942377  0.854312  0.943366              0.967034                 0.948248          0.934481           0.932846         0.979544        0.979186
DINO (ViT-L) [val]   0.960038  0.910843  0.959877              0.975013          

## Parte 3 — Cascada clínica completa: B3→B4 VS ViT-L→ViT-L

In [5]:

# === PARTE 3: COMPARACIÓN DE LA CASCADA CLÍNICA sobre una muestra del test ===
# Usa el motor real (ClinicalInferenceEngine) con TODOS los calibradores de cada
# modelo: triaje -> especialista -> estado clínico, conformal, threshold por sitio.

df_test = pd.read_csv("data/processed/04_test.csv")

# Muestra estratificada ~50/50 benigno/maligno (sobre-representa malignos para
# tener señal en "melanomas perdidos"; el test real tiene ~2% de malignos).
mal = df_test[df_test["benign_malignant"] == "malignant"]
ben = df_test[df_test["benign_malignant"] == "benign"]
n_each = N_CASCADE_SAMPLE // 2
samp = pd.concat([
    mal.sample(min(n_each, len(mal)), random_state=SEED),
    ben.sample(min(n_each, len(ben)), random_state=SEED),
]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"Muestra cascada: {len(samp)} imagenes "
      f"({int((samp['benign_malignant'] == 'malignant').sum())} malignas)")

PROJECT_ROOT = "/content/SKIN-PROYECT"
def resolve_path(p):
    marker = "data/raw/"
    if marker in p:
        p = p[p.index(marker):]
    return os.path.join(PROJECT_ROOT, p)

def build_engine(meta):
    f = meta["folder"]
    return ClinicalInferenceEngine(
        binary_model_path=meta["binary"],
        multiclass_model_path=meta["multiclass"],
        temperature_binary_path=opt(os.path.join(f, "temperature_binary.pt")),
        temperature_multiclass_path=opt(os.path.join(f, "temperature_multiclass.pt")),
        conformal_binary_path=opt(os.path.join(f, "conformal_binary.pt")),
        conformal_multiclass_path=opt(os.path.join(f, "conformal_multiclass.pt")),
        risk_control_binary_path=opt(os.path.join(f, "conformal_risk_binary.pt")),
        per_site_threshold_path=opt(os.path.join(f, "per_site_threshold_binary.pt")),
    )

casc = {}
for name, meta in MODEL_META.items():
    if meta["multiclass"] is None:
        print(f"\n> Cascada {name}: sin multiclase, se omite.")
        continue
    print(f"\n> Cascada: {name}")
    engine = build_engine(meta)
    states, lost, covered, n_mal = {}, 0, 0, 0
    for _, r in tqdm(samp.iterrows(), total=len(samp), desc=f"casc:{name}", leave=False):
        rep = engine.infer(
            resolve_path(r["image_path"]),
            age=r.get("age"), sex=r.get("sex"), anatom_site=r.get("anatom_site"),
            conformal=True, risk_control=True)
        st = rep["status"]
        states[st] = states.get(st, 0) + 1
        is_mal = (r["benign_malignant"] == "malignant")
        if is_mal:
            n_mal += 1
            if st == "GREEN":
                lost += 1          # maligno clasificado benigno = melanoma perdido
        cs = rep.get("conformal_set_binary")
        truth = "maligno" if is_mal else "benigno"
        if cs and truth in cs:
            covered += 1
    del engine
    if device.type == "cuda":
        torch.cuda.empty_cache()
    recall_casc = (n_mal - lost) / n_mal if n_mal else float("nan")
    casc[name] = dict(
        states=states, malignos=int(n_mal), melanomas_perdidos=int(lost),
        recall_derivacion=float(recall_casc),
        cobertura_conformal=float(covered / len(samp)), n=int(len(samp)),
    )
    print(f"  estados={states}")
    print(f"  malignos perdidos (GREEN)={lost}/{n_mal} | "
          f"recall derivacion={recall_casc:.4f} | "
          f"cobertura conformal={covered / len(samp):.3f}")

# --- Tabla comparativa cascada ---
if casc:
    print("\n" + "=" * 72 + "\nCOMPARATIVA CASCADA CLINICA\n" + "=" * 72)
    tabla = {k: {
        "malignos": v["malignos"],
        "perdidos": v["melanomas_perdidos"],
        "recall_derivacion": round(v["recall_derivacion"], 4),
        "cobertura_conformal": round(v["cobertura_conformal"], 3),
    } for k, v in casc.items()}
    print(pd.DataFrame(tabla).T.to_string())
    with open(os.path.join(OUT_DIR, "comparison_cascade.json"), "w") as f:
        json.dump(dict(config=dict(n_sample=int(len(samp)), tta="engine TTA-6"),
                       per_model=casc), f, indent=2, default=str)
    print(f"\nGuardado: {OUT_DIR}/comparison_cascade.json")
else:
    print("\nAVISO: ningun modelo con cascada completa; Parte 3 omitida.")

Muestra cascada: 400 imagenes (200 malignas)

> Cascada: B34 (CNN)
Inicializando motor clínico en cuda...
Cargando Nivel 1: Triaje Binario (High Recall)...
Cargando Nivel 2: Especialista Multiclase (High Precision)...
   Temperatura binaria cargada (scalar): T=1.190
   Temperatura multiclase cargada (vector, C=8): T=[1.589, 1.488, 1.481, 1.512, 1.259, 1.55, 1.451, 1.568]
   Conformal binario cargado: alpha=0.03, n_calib=42997
   Conformal multiclase cargado: alpha=0.1, n_calib=2983
   RiskControl binario cargado: target_fnr=0.05, threshold=0.2948, n_calib_pos=995 n_calib_neg=42002


casc:B34 (CNN):   0%|          | 0/400 [00:00<?, ?it/s]

  estados={'GREEN': 189, 'RED': 49, 'CRITICAL': 155, 'AMBER_PRECANCEROUS': 6, 'AMBER': 1}
  malignos perdidos (GREEN)=3/200 | recall derivacion=0.9850 | cobertura conformal=0.970

> Cascada: DINO (ViT-L)
Inicializando motor clínico en cuda...
Cargando Nivel 1: Triaje Binario (High Recall)...
Cargando Nivel 2: Especialista Multiclase (High Precision)...
   Temperatura binaria cargada (scalar): T=1.373
   Temperatura multiclase cargada (vector, C=8): T=[1.673, 1.533, 1.606, 1.615, 1.399, 1.61, 1.469, 1.55]
   Conformal binario cargado: alpha=0.03, n_calib=42997
   Conformal multiclase cargado: alpha=0.1, n_calib=2983
   RiskControl binario cargado: target_fnr=0.05, threshold=0.5939, n_calib_pos=995 n_calib_neg=42002


casc:DINO (ViT-L):   0%|          | 0/400 [00:00<?, ?it/s]

  estados={'GREEN': 191, 'RED': 37, 'CRITICAL': 164, 'AMBER_PRECANCEROUS': 7, 'AMBER': 1}
  malignos perdidos (GREEN)=2/200 | recall derivacion=0.9900 | cobertura conformal=0.968

COMPARATIVA CASCADA CLINICA
              malignos  perdidos  recall_derivacion  cobertura_conformal
B34 (CNN)        200.0       3.0              0.985                0.970
DINO (ViT-L)     200.0       2.0              0.990                0.968

Guardado: /content/drive/MyDrive/SKIN-PROYECT/evaluation/comparison_cascade.json


In [ ]:
# === Occlusion ALTA RESOLUCIÓN ===
import os, numpy as np, cv2, pandas as pd, matplotlib, torch
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display
from src.inference.predict import ClinicalInferenceEngine
from src.inference.explainability import OcclusionSensitivityExplainer

FOLDER  = "models/DINO-V2"     # carpeta del modelo a explicar (models/B34 = CNN, models/DINO-V2 = ViT)
DIAG    = "Melanoma"       # lesión a explicar
IMG_IDX = 50               # índice de la lesión a visualizar
OUT = "/content/drive/MyDrive/SKIN-PROYECT/figuras"; os.makedirs(OUT, exist_ok=True)

def opt(p): return p if (p and os.path.exists(p)) else None
def resolve_img(rel):
    if "data/raw/" in rel: rel = rel[rel.index("data/raw/"):]
    for root in ("/content/SKIN-PROYECT", "/content", "."):
        c = os.path.join(root, rel)
        if os.path.exists(c): return c
    return rel

eng = ClinicalInferenceEngine(
    binary_model_path=f"{FOLDER}/best_binary_model_SOUP.pth",
    multiclass_model_path=f"{FOLDER}/best_multiclass_model_SOUP.pth",
    temperature_binary_path=opt(f"{FOLDER}/temperature_binary.pt"))

df = pd.read_csv("data/processed/07_test_multiclass.csv")
row = df[df["diagnosis"] == DIAG].reset_index(drop=True).iloc[IMG_IDX]
img_path = resolve_img(row["image_path"])
age, sex, site = row.get("age"), row.get("sex"), row.get("anatom_site")

img_bin, meta = eng.preprocess(img_path, age, sex, site, is_multiclass=False)
H = img_bin.shape[2]
orig = cv2.resize(cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB), (448, 448))

ex = OcclusionSensitivityExplainer(eng.triaje, patch_size=H//14, stride=max(1, H//28), batch_size=64)
hm, pred = ex.explain(img_bin, meta)
hm = cv2.resize(hm.astype(np.float32), (448, 448))
hm = cv2.GaussianBlur(hm, (0, 0), sigmaX=9)
hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)

fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
ax[0].imshow(orig); ax[0].set_title("Original"); ax[0].axis("off")
ax[1].imshow(orig); ax[1].imshow(hm, cmap="jet", alpha=0.5)
ax[1].set_title(f"Occlusion alta-res · P={pred['prob']:.2f}"); ax[1].axis("off")
fig.tight_layout()
out = f"{OUT}/occlusion_hires.png"; fig.savefig(out, dpi=150, bbox_inches="tight"); plt.close(fig)
display(IPImage(filename=out))


Inicializando motor clínico en cuda...
Cargando Nivel 1: Triaje Binario (High Recall)...
Cargando Nivel 2: Especialista Multiclase (High Precision)...
